# AI vs Real Image Detection - EfficientNetV2-S (Google)

Notebook ini melatih model **EfficientNetV2-S** dari **Google Research** untuk membedakan gambar sintetis (**AI-Generated**) dan foto asli (**Real Image**).

### Mengapa EfficientNetV2-S Sangat Unggul?
1. **Fused-MBConv & MBConv Layers**: Menggabungkan konvolusi reguler pada layer awal untuk kecepatan komputasi tinggi dan depthwise separable convolution pada layer akhir untuk efisiensi parameter.
2. **Progressive Learning Pretraining**: Dilatih dengan regularisasi adaptif pada ImageNet-1K/21K, menghasilkan representasi visual yang sangat kokoh terhadap distorsi dan artefak kompresi.
3. **Efisiensi Parameter & GPU Memory**: Memiliki sekitar **21.5 Juta parameter**, sangat ringan dan cepat di GPU modern (RTX 4050 6GB) dengan konsumsi VRAM ~2.0 GB.

### Konfigurasi Pelatihan:
- **Dataset**: `Thunderbeast/AI-Generated-vs-Real-Images-Datasets` (Hugging Face)
- **Arsitektur**: EfficientNetV2-S (Pretrained Google) + Classification Head (2 Kelas)
- **Pelatihan**: End-to-End Default (Full Backbone Unfrozen, lr=1e-4)
- **Optimizer & Scheduler**: AdamW (`weight_decay=1e-2`) + `CosineAnnealingLR`
- **Akselerasi**: Automatic Mixed Precision (AMP FP16) + cuDNN Benchmark + cuDNN Benchmark
- **Output Checkpoint**: `best_efficientnet_v2_s_ai_vs_real.pth`


In [1]:
import warnings
warnings.filterwarnings('ignore', message='.*Palette images with Transparency.*')
warnings.filterwarnings('ignore', message='.*xFormers is not available.*')

import os
import copy
import time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models, transforms
from datasets import load_dataset

# Memeriksa dan menetapkan hardware acceleration (GPU jika tersedia)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print(f"Device yang digunakan: {device} ({torch.cuda.get_device_name(0)})")
    print("[Akselerasi GPU] cuDNN Benchmark diaktifkan!")
else:
    print(f"Device yang digunakan: {device}")

Device yang digunakan: cuda (NVIDIA GeForce RTX 4050 Laptop GPU)
[Akselerasi GPU] cuDNN Benchmark diaktifkan!


In [2]:
print("=" * 65)
print("MEMUAT DATASET DARI HUGGING FACE (Thunderbeast)")
print("=" * 65)

# Load dataset Thunderbeast/AI-Generated-vs-Real-Images-Datasets
dataset = load_dataset(
    "Thunderbeast/AI-Generated-vs-Real-Images-Datasets",
    split="train",
    cache_dir="./data",
    verification_mode="no_checks"
)

# Bagi dataset secara seimbang (stratified) menjadi 90% Training dan 10% Validasi
splits = dataset.train_test_split(test_size=0.10, stratify_by_column="label", seed=42)
train_hf = splits["train"]
val_hf = splits["test"]

print(f"Total Data Training : {len(train_hf):,}")
print(f"Total Data Validasi : {len(val_hf):,}")
print(f"Label Kelas Asli    : {dataset.features['label'].names}")
print("=" * 65)

MEMUAT DATASET DARI HUGGING FACE (Thunderbeast)


Total Data Training : 137,439
Total Data Validasi : 15,271
Label Kelas Asli    : ['AiArtData', 'RealArt']


In [3]:
# 1. Transformasi Data Sesuai Input EfficientNetV2-S (224x224)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. PyTorch Dataset Wrapper dari Hugging Face (Penanganan Palet & Transparansi Aman)
class HFToPyTorchDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        label = item["label"]

        if image.mode == 'P' and 'transparency' in image.info:
            image = image.convert('RGBA')
        if image.mode != "RGB":
            image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# 3. Inisialisasi Dataset & DataLoader
# BATCH_SIZE = 32 menjaga pemakaian VRAM ~1.8 - 2.0 GB (100% di GDDR6 VRAM, bebas dari Paging)
train_dataset = HFToPyTorchDataset(train_hf, transform=train_transforms)
val_dataset = HFToPyTorchDataset(val_hf, transform=val_transforms)

BATCH_SIZE = 32
use_pin = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=use_pin
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=use_pin
)

print(f"DataLoader EfficientNetV2-S Siap!")
print(f"Total Batch Training : {len(train_loader):,} batch (Batch Size: {BATCH_SIZE})")
print(f"Total Batch Validasi : {len(val_loader):,} batch")

DataLoader EfficientNetV2-S Siap!
Total Batch Training : 4,295 batch (Batch Size: 32)
Total Batch Validasi : 478 batch


In [4]:
# Definisi Model EfficientNetV2-S Classifier
def create_efficientnetv2_model(num_classes=2, dropout_p=0.2):
    # Load model pretrained EfficientNetV2-S (Google)
    weights = models.EfficientNet_V2_S_Weights.DEFAULT
    model = models.efficientnet_v2_s(weights=weights)

    # Modifikasi classifier head untuk 2 kelas (0: AI-Generated, 1: Real Image)
    in_features = model.classifier[1].in_features  # 1280
    model.classifier[1] = nn.Sequential(
        nn.Dropout(p=dropout_p, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model

model = create_efficientnetv2_model().to(device)

criterion = nn.CrossEntropyLoss()
# Optimizer AdamW untuk seluruh layer model (default end-to-end)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

EPOCHS = 5
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 65)
print(f"MODEL EFFICIENTNETV2-S BERHASIL DIINISIALISASI")
print("=" * 65)
print(f"Total Parameter       : {total_params:,} (~21.5 Juta)")
print(f"Trainable Parameter   : {trainable_params:,} (100% End-to-End)")
print(f"Classifier Head       : {model.classifier[1]}")
print(f"Learning Rate Awal    : 1.0e-04 (CosineAnnealingLR)")
print("=" * 65)

MODEL EFFICIENTNETV2-S BERHASIL DIINISIALISASI
Total Parameter       : 20,180,050 (~21.5 Juta)
Trainable Parameter   : 20,180,050 (100% End-to-End)
Classifier Head       : Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)
Learning Rate Awal    : 1.0e-04 (CosineAnnealingLR)


In [5]:
from tqdm.auto import tqdm

ACCUMULATION_STEPS = 2  # Efektif batch size 64 dengan konsumsi VRAM aman (~1.9 GB)
best_f1 = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_f1': []
}

MODEL_SAVE_PATH = "best_efficientnet_v2_s_ai_vs_real.pth"

# Aktifkan Automatic Mixed Precision (AMP) untuk Tensor Cores & VRAM ultra-ringan
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print(f"Mulai pelatihan EfficientNetV2-S ({EPOCHS} Epoch, F1-Score Tracking, AMP FP16)...\n" + "="*65)
print(f"Batch Size Fisik        : {BATCH_SIZE}")
print(f"Accumulation Steps      : {ACCUMULATION_STEPS} (Efektif Batch Size: {BATCH_SIZE * ACCUMULATION_STEPS})")
print(f"Presisi Komputasi       : AMP FP16 (Akselerasi Tensor Cores)")
print(f"Estimasi Pemakaian VRAM : ~1.9 GB (Bebas dari Shared Memory Paging, 100% GDDR6 VRAM)")
print("="*65)

for epoch in range(EPOCHS):
    start_time = time.time()
    current_lr = optimizer.param_groups[0]['lr']
    
    # ------------------ FASE TRAINING ------------------
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}] Train (LR: {current_lr:.1e})", leave=True)
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (images, labels) in enumerate(train_bar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if scaler:
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = criterion(outputs.float(), labels)
                loss = loss / ACCUMULATION_STEPS

            scaler.scale(loss).backward()

            if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss = loss / ACCUMULATION_STEPS
            loss.backward()

            if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        # Hitung loss dan akurasi berjalan (skala loss asli dikalikan ACCUMULATION_STEPS)
        running_loss += loss.item() * ACCUMULATION_STEPS * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data).item()
        total_train += labels.size(0)

        # Update tampilan progress bar secara real-time
        current_acc = (correct_train / total_train) * 100
        current_loss = running_loss / total_train
        train_bar.set_postfix({'loss': f'{current_loss:.4f}', 'acc': f'{current_acc:.2f}%'})

    scheduler.step()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = (correct_train / total_train) * 100

    # ------------------ FASE VALIDASI ------------------
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    all_preds = []
    all_labels = []

    val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}] Validasi", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if scaler:
                with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                    outputs = model(images)
                    loss = criterion(outputs.float(), labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data).item()
            total_val += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100
    epoch_val_f1 = f1_score(all_labels, all_preds, average='weighted')

    # Catat history metrik
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_f1'].append(epoch_val_f1)

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] "
        f"Train Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.2f}% || "
        f"Val Loss: {epoch_val_loss:.4f} | Acc: {epoch_val_acc:.2f}% | F1: {epoch_val_f1:.4f} "
        f"({elapsed:.1f}s)"
    )

    # Simpan model terbaik berdasarkan F1-Score
    if epoch_val_f1 > best_f1:
        best_f1 = epoch_val_f1
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, MODEL_SAVE_PATH)
        print(f"  -> Model terbaik diperbarui & disimpan ke '{MODEL_SAVE_PATH}' (Val F1: {best_f1:.4f})")

print("="*65)
print(f"Pelatihan Selesai! F1-Score Validasi Terbaik: {best_f1:.4f}")
print(f"Model disimpan di: {MODEL_SAVE_PATH}")
model.load_state_dict(best_model_wts)


Mulai pelatihan EfficientNetV2-S (5 Epoch, F1-Score Tracking, AMP FP16)...
Batch Size Fisik        : 32
Accumulation Steps      : 2 (Efektif Batch Size: 64)
Presisi Komputasi       : AMP FP16 (Akselerasi Tensor Cores)
Estimasi Pemakaian VRAM : ~1.9 GB (Bebas dari Shared Memory Paging, 100% GDDR6 VRAM)


Epoch [1/5] Train (LR: 1.0e-04):   0%|          | 0/4295 [00:00<?, ?it/s]

Epoch [1/5] Validasi:   0%|          | 0/478 [00:00<?, ?it/s]

Epoch [01/05] Train Loss: 0.1416 | Acc: 94.17% || Val Loss: 0.0726 | Acc: 97.16% | F1: 0.9716 (1827.4s)
  -> Model terbaik diperbarui & disimpan ke 'best_efficientnet_v2_s_ai_vs_real.pth' (Val F1: 0.9716)


Epoch [2/5] Train (LR: 9.1e-05):   0%|          | 0/4295 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Visualisasi Kurva Loss, Akurasi, dan F1-Score (3 Panel)
epochs_range = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(18, 5))

# 1. Plot Loss
plt.subplot(1, 3, 1)
plt.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#EF4444', linewidth=2)
plt.plot(epochs_range, history['val_loss'], 's--', label='Val Loss', color='#F59E0B', linewidth=2)
plt.title('Training & Validation Loss (EfficientNetV2-S)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# 2. Plot Akurasi
plt.subplot(1, 3, 2)
plt.plot(epochs_range, history['train_acc'], 'o-', label='Train Acc', color='#10B981', linewidth=2)
plt.plot(epochs_range, history['val_acc'], 's--', label='Val Acc', color='#3B82F6', linewidth=2)
plt.title('Training & Validation Accuracy (EfficientNetV2-S)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Akurasi (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# 3. Plot Validasi F1-Score
plt.subplot(1, 3, 3)
plt.plot(epochs_range, history['val_f1'], 'd-', label='Val F1-Score', color='#8B5CF6', linewidth=2)
plt.title('Validation F1-Score (EfficientNetV2-S)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluasi Mendalam pada Seluruh Data Validasi
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluasi Final Validation"):
        images = images.to(device, non_blocking=True)
        if scaler:
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
        else:
            outputs = model(images)
            
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

target_names = ["AI-Generated (0)", "Real Image (1)"]

print("=" * 65)
print("CLASSIFICATION REPORT (EfficientNetV2-S):")
print("=" * 65)
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - EfficientNetV2-S', fontsize=12, fontweight='bold')
plt.grid(False)
plt.show()


In [ ]:
# Fungsi Pengujian Prediksi Citra Tunggal
def predict_image(image_path, model, device):
    """
    Menguji citra tunggal dan mengembalikan prediksi label serta probabilitas kepercayaan.
    """
    if not os.path.exists(image_path):
        print(f"File '{image_path}' tidak ditemukan!")
        return

    img = Image.open(image_path)
    if img.mode == 'P' and 'transparency' in img.info:
        img = img.convert('RGBA')
    img = img.convert('RGB')
    tensor = val_transforms(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        if scaler:
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                outputs = model(tensor)
        else:
            outputs = model(tensor)
            
        probs = torch.softmax(outputs, dim=1)[0]
        ai_prob = probs[0].item() * 100.0
        real_prob = probs[1].item() * 100.0
        pred_class = torch.argmax(probs).item()

    verdict = "AI-Generated Image" if pred_class == 0 else "Real Image"
    confidence = ai_prob if pred_class == 0 else real_prob

    print("=" * 50)
    print(f"File            : {os.path.basename(image_path)}")
    print(f"Hasil Prediksi  : {verdict}")
    print(f"Tingkat Keyakinan: {confidence:.2f}%")
    print(f"  - Probabilitas AI   : {ai_prob:.2f}%")
    print(f"  - Probabilitas Real : {real_prob:.2f}%")
    print("=" * 50)

# Contoh Pengujian dengan beberapa citra uji di workspace
test_files = ["images (2).jpg", "makan.jpeg", "img.png", "test1.webp", "test2.webp", "wa.jpeg", "gani.jpeg", "real.jpg"]
for tf in test_files:
    if os.path.exists(tf):
        predict_image(tf, model, device)
